In [0]:
%run "./01_core_functions"

# Standardization Engine - Bronze → Silver

Given a Bronze source table, this notebook applies the full metadata-driven
standardization pipeline and writes the result to its mapped Silver table.

**Pipeline steps:**
1. Load & validate metadata for the source tables
2. Read the Bronze table
3. Standardize column names to snake_case
4. Rename → fill missing → cast → reorder columns, in a single pass
5. Cleanse string columns (trim + blank → null)
6. Add audit columns
7. Deduplicate (key on metadata primary key columns, if defined)
8. Run data quality checks on mandatory columns
9. Write to Silver (with schema evolution support)

## Main entry point: `standardize_table`

Call this once per Bronze source tables

In [0]:
def standardize_table(src_catalog, src_schema, src_table, batch_id=None):
    """
    Run the full Bronze -> Silver standardization pipeline for one source table

    Args:
        src_catalog, src_schema, src_table: identifiers of the Bronze source table
        batch_id: optional shared batch id

    Returns:
        The final Silver DataFrame

    Raises:
        Exception on missing metadata, read/write failures, or failed data
        quality checks. The exception message identifies the source table and
        the step that failed.
    """

    source = f"{src_catalog}.{src_schema}.{src_table}"

    try:
        
        # Metadata
        mapping_df = get_metadata(src_catalog, src_schema, src_table)
        metadata = mapping_df.collect()

        if len(metadata) == 0:
            raise Exception(f"Metadata not found for {source}")

        # Read Bronze

        log(f"Reading {source}")
        df = spark.table(source)

        # Snake case incoming column names
        
        df = df.toDF(*[snake_case(c) for c in df.columns])
        available_cols = set(df.columns)

       
        # Rename + fill missing + cast + reorder — single pass
        select_exprs = []
        for r in metadata:
            src_col = r.src_column.lower()
            if src_col in available_cols:
                col_expr = F.col(src_col).cast(r.tgt_dtype)
            else:
                col_expr = F.lit(r.default_value).cast(r.tgt_dtype)
            select_exprs.append(col_expr.alias(r.tgt_column))

        df = df.select(*select_exprs)

        
        # Cleanse strings (trim + blank -> null)
        
        df = trim_string_columns(df)

        
        # Audit columns
        df = add_audit_columns(df, src_catalog, src_schema, src_table, batch_id=batch_id)

        
        # Deduplicate (only if metadata defines primary key columns)
        
        pk_columns = [r.tgt_column for r in metadata if getattr(r, "is_primary_key", False)]
        if pk_columns:
            log(f"Deduplicating on key columns: {pk_columns}")
            df = deduplicate(df, key_columns=pk_columns)
        else:
            df = df.dropDuplicates()
            log("No primary key columns defined in metadata - deduplicating based on all columns")


        # Write in silver layer
        target = metadata[0]
        target_table = f"{target.tgt_catalog}.{target.tgt_schema}.{target.tgt_table}"

        partition_cols = [r.tgt_column for r in metadata if getattr(r, "partition_column", False)]

        log(f"Writing {target_table}" + (f" partitioned by {partition_cols}" if partition_cols else ""))

        writer = (
            df.write
            .format("delta")
            .mode(WRITE_MODE)
            .option("mergeSchema", "true")
        )

        if partition_cols:
            writer = writer.partitionBy(*partition_cols)

        writer.saveAsTable(target_table)

        log(f"Completed {source} -> {target_table}")

        return df

    
    except Exception as e:
        log(f"FAILED processing {source}: {e}", level="ERROR")
        raise